In [54]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from linear_combination import generate_mixture_spectra
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Set a nice default style
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

In [55]:
df = pd.read_excel('Spectral_library_clean.xlsx')
df.ffill(axis=0, inplace=True)
df.bfill(axis=0, inplace=True)  # handles leading NaNs

wavelength = df['Wavelength']
abs_spectra = df.iloc[:, 1:].to_numpy().T

In [56]:
df_mix_spec = pd.read_excel("Spectral_library_with_mixtures.xlsx")
df_mix_spec.ffill(axis=0, inplace=True)
df_mix_spec.bfill(axis=0, inplace=True)

N_species = abs_spectra.shape[0]
N_mix = df_mix_spec.shape[1]

df_mix_weights = pd.read_excel("Spectral_library_mixture_weights.xlsx")

mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

In [57]:
species_cols = list(df_mix_weights.columns[1:])
mixture_cols = df_mix_weights["mixture_name"].tolist()
mix_spectra = df_mix_spec[mixture_cols].to_numpy().T

df_mixtures = pd.DataFrame(
    data=mix_spectra.T,
    columns=mixture_cols
)

wavelength_mix = df_mix_spec["Wavelength"].to_numpy()
df_mixtures.insert(0, "Wavelength", wavelength_mix)

In [58]:
X = df_mix_spec[mixture_cols].to_numpy().T
Y = df_mix_weights[species_cols].to_numpy()

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.9, random_state=0)

noise_level = 0.01

X_train_noisy = X_train + np.random.normal(0, noise_level, X_train.shape)
Y_train_noisy = Y_train + np.random.normal(0, noise_level, Y_train.shape)
X_test_noisy = X_test + np.random.normal(0, noise_level, X_test.shape)

In [59]:
pls = PLSRegression(n_components=N_species)

# Train the model
pls.fit(X_train_noisy, Y_train)

Y_pred_raw = pls.predict(X_test_noisy)

Y_pred = np.clip(Y_pred_raw, 0, None)
row_sums = Y_pred.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
Y_pred_clean = Y_pred / row_sums

mae = mean_absolute_error(Y_test, Y_pred_clean)
print(f"Overall Mean Absolute Error: {mae:.4f}")

for i in range(len(Y_test)):
    print(f"Test Mixture {i+1}:")
    print(f"  True : {np.round(Y_test[i], 3)}")
    print(f"  PLS  : {np.round(Y_pred_clean[i], 3)}")

Overall Mean Absolute Error: 0.0661
Test Mixture 1:
  True : [0.213 0.    0.    0.162 0.624 0.    0.    0.    0.   ]
  PLS  : [0.192 0.    0.109 0.003 0.435 0.077 0.165 0.    0.019]
Test Mixture 2:
  True : [0.    0.    0.    0.    0.    0.    0.253 0.504 0.243]
  PLS  : [0.    0.    0.069 0.06  0.    0.008 0.201 0.432 0.231]
Test Mixture 3:
  True : [0.    0.    0.    0.    0.214 0.741 0.045 0.    0.   ]
  PLS  : [0.087 0.    0.    0.    0.299 0.601 0.013 0.    0.   ]
Test Mixture 4:
  True : [0.    0.651 0.    0.    0.151 0.198 0.    0.    0.   ]
  PLS  : [0.    0.38  0.017 0.272 0.165 0.144 0.    0.021 0.   ]
Test Mixture 5:
  True : [0.    0.    0.    0.    0.    0.    0.844 0.126 0.03 ]
  PLS  : [0.    0.    0.115 0.027 0.12  0.    0.485 0.24  0.014]
Test Mixture 6:
  True : [0.236 0.    0.431 0.    0.    0.    0.    0.    0.333]
  PLS  : [0.222 0.065 0.256 0.124 0.024 0.    0.03  0.    0.279]
Test Mixture 7:
  True : [0.    0.    0.856 0.012 0.    0.    0.    0.131 0.   ]
  PLS  